In [ ]:
async def process_response(url, report_idx, event_id, session: aiohttp.ClientSession):
	basic_response = report_idx, None
	try:
		async with session.get(url, headers=headers, timeout=5) as response:
			if response.status != 200:
				raise Exception(f"Response status: {response.status}")

			page_html = await response.text()
			parsed_response = BeautifulSoup(page_html, "html.parser")
			date = parsed_response.find("time", {"datetime": True})
			if date is None:
				date = parsed_response.find(
					"span", {"class": re.compile("^.*(?:date|time|datetime).*$")}
				)
				# date = parsed_response.find_all("span", {"class": re.compile("^.*(?:date|time|datetime).*$")})
				# date = [d.text for d in date]
				date = date.text if date is not None else date
			else:
				date = date["datetime"]

			return report_idx, date
	except Exception as url_exception:
		print("failed first URL", url_exception, url)
		if CHECK_WAYBACK is True:
			try:
				response = await reach_wayback(url, session)
				if response is None:
					raise Exception
				return report_idx, response
			except Exception as e:
				report_unavailable_links[event_id] += [url]
				return basic_response
		return basic_response

In [ ]:
from tqdm import tqdm
from bs4 import BeautifulSoup
from collections import defaultdict
import requests
import re
import asyncio
import aiohttp
import tenacity
import os
import json

headers = {
	"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:146.0) Gecko/20100101 Firefox/146.0",
	# "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
	"Accept-Language": "en-US,en;q=0.5",
	"Accept-Encoding": "gzip",
	"Connection": "keep-alive",
	"Upgrade-Insecure-Requests": "1",
	"Sec-Fetch-Dest": "document",
	"Sec-Fetch-Mode": "navigate",
	"Sec-Fetch-Site": "none",
	"Sec-Fetch-User": "?1",
}

report_unavailable_links = defaultdict(list)

CHECK_WAYBACK = True

WAYBACK_URL = "http://web.archive.org/cdx/search/cdx?url={url}&collapse=digest"


@tenacity.retry(
	stop=tenacity.stop_after_attempt(2), wait=tenacity.wait_random_exponential(multiplier=1, max=10)
)
async def reach_wayback(url, session):
	search_url = WAYBACK_URL.format(url=url)
	async with session.get(search_url, headers=headers) as response:
		response = await response.text()
		if response == "\n":  # no snapshots found
			return None
		return response.split("\n")[0].split(" ")[1]


@tenacity.retry(
	stop=tenacity.stop_after_attempt(3), wait=tenacity.wait_random_exponential(multiplier=1, max=10)
)
async def reach_url(url, session):
	try:
		async with session.get(url, headers=headers) as response:
			if response.status != 200:
				raise Exception(f"Response status: {response.status}")

			return await response.content.read()
	except Exception as e:
		# print("Reaching URL error", e)
		return None


async def process_response(url, report_idx, event_id, session: aiohttp.ClientSession):
	basic_response = report_idx, None, "NOT WORKING---" + url
	try:
		page_html = await reach_url(url, session)
		if page_html is None:
			raise Exception("Could not reach link")

		parsed_response = BeautifulSoup(page_html, "html.parser")
		date = parsed_response.find("time", {"datetime": True})
		if date is None:
			# date = parsed_response.find("span", {"class": re.compile("^.*(?:date|time|datetime).*$")})
			# date = date.text if date is not None else date
			date = parsed_response.find_all(
				"span", {"class": re.compile("^.*(?:date|time|datetime).*$")}
			)
			date = [d.text for d in date]
			date: list[str] = None if len(date) < 1 else date

			invalid_years = ["2023", "2024", "2025", "2026"]
			for d in date:
				for y in invalid_years:
					if y in d:
						raise Exception("Link is too new or something has changed")
		else:
			date = date["datetime"]

		return report_idx, date, url
	except Exception as url_exception:
		# print("failed first URL", url_exception, url)
		if CHECK_WAYBACK is True:
			try:
				response = await reach_wayback(url, session)
				if response is None:
					raise Exception

				parsed_response = BeautifulSoup(response, "html.parser")
				date = parsed_response.find("time", {"datetime": True})
				if date is None:
					# date = parsed_response.find("span", {"class": re.compile("^.*(?:date|time|datetime).*$")})
					# date = date.text if date is not None else date
					date = parsed_response.find_all(
						"span", {"class": re.compile("^.*(?:date|time|datetime).*$")}
					)
					date = [d.text for d in date]
					date = None if len(date) < 1 else date
				else:
					date = date["datetime"]
				return report_idx, date, WAYBACK_URL.format(url=url)
			except Exception as e:
				# print("Failed Wayback URL", e, url)
				report_unavailable_links[event_id] += [url]
				return basic_response
		return basic_response


async def get_dates_for_claim(claim, date_path):
	event_id = claim["event_id"]
	report_dates = defaultdict(list)
	todo_requests = []
	output_file_path = os.path.join(date_path, f"{event_id}.json")
	if f"{event_id}.json" in os.listdir(date_path):
		return
	async with aiohttp.ClientSession() as session:
		for report_idx, report in enumerate(claim["reports"]):
			url = report["link"]
			todo_requests += [
				asyncio.create_task(process_response(url, report_idx, event_id, session))
			]
		report_dates[event_id] = await asyncio.gather(*todo_requests, return_exceptions=True)
	json.dump(report_dates, open(output_file_path, "w"), indent=2)


async def get_all_dates(data, output_dir):
	os.makedirs(output_dir, exist_ok=True)

	for claim in tqdm(data[:]):
		# str -> list[str]
		claim = json.load(open(claim, "r")) if isinstance(claim, str) else claim
		await get_dates_for_claim(claim, output_dir)

# LIAR-RAW

In [34]:
splits = ["train", "test", "val"]
for split in splits:
	data_file = f"../datasets/data/LIAR-RAW/{split}.json"
	output_dir = f"dates/LIAR-RAW_{split}"
	data = json.load(open(data_file, "r"))
	await get_all_dates(data, output_dir)

  3%|▎         | 352/10065 [30:02<21:24:18,  7.93s/it]/tmp/ipykernel_83777/2080769639.py:59: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  parsed_response = BeautifulSoup(page_html, "html.parser")
 24%|██▎       | 2389/10065 [3:50:40<8:53:13,  4.17s/it]  Some characters could not be decoded, and were replaced with REPLA

In [ ]:
data[0]

# RAWFC

In [ ]:
splits = ["train", "test", "val"]
### https://archive.ph/ alternative
for split in splits[:]:
	data_file = f"../datasets/data/RAWFC/{split}"
	output_dir = f"dates/RAWFC_{split}"
	data = [os.path.join(data_file, file) for file in os.listdir(data_file)]
	await get_all_dates(data, output_dir)

In [95]:
interest_url = "https://www.facebook.com/pages/category/Community/Fallen-Hero-Arminas-Pileckas-Brave-son-of-Europa/1717873738459987/"
# https://www.facebook.com/people/Fallen-Hero-Arminas-Pileckas-Brave-son-of-Europa/100071517973689/

r = requests.get(interest_url, headers=headers)

In [ ]:
WAYBACK_URL = "http://web.archive.org/cdx/search/cdx?url={url}"
url = "https://www.fr24news.com/a/2020/08/did-putins-daughter-die-after-taking-the-covid-19-vaccine.html"
input_url = WAYBACK_URL.format(url=url)

response = requests.get(input_url, headers={**headers, "Accept-Encoding": "gzip"})
response.status_code

200

In [28]:
response.text

'com,fr24news)/a/2020/08/did-putins-daughter-die-after-taking-the-covid-19-vaccine.html 20200826011112 https://www.fr24news.com/a/2020/08/did-putins-daughter-die-after-taking-the-covid-19-vaccine.html text/html 200 5QDH23Q3RNQJ5MC7PAZPPIDL3G7DPOIA 21676\ncom,fr24news)/a/2020/08/did-putins-daughter-die-after-taking-the-covid-19-vaccine.html 20250818191607 https://www.fr24news.com/a/2020/08/did-putins-daughter-die-after-taking-the-covid-19-vaccine.html text/html 200 HDYC2KBOBFMRQXVZKXAW6RDEGEHBJA26 7640\n'

In [64]:
wayback_endpoint = f"http://web.archive.org/cdx/search/cdx?url={interest_url}"

r = requests.get(wayback_endpoint, headers=headers)

In [69]:
wayback_endpoint

'http://web.archive.org/cdx/search/cdx?url=https://outrageous-injustice.webnode.se/muslims/'

In [65]:
r.status_code

200

In [73]:
print(r.content.decode())

se,webnode,outrageous-injustice)/muslims 20171225171214 http://outrageous-injustice.webnode.se:80/muslims text/html 200 7IZDBNGHAX3ID77LIHONF6VXXWT2XDD6 76943
se,webnode,outrageous-injustice)/muslims 20180125191945 http://outrageous-injustice.webnode.se:80/muslims text/html 302 3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ 334
se,webnode,outrageous-injustice)/muslims 20180226035137 http://outrageous-injustice.webnode.se:80/muslims text/html 302 3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ 334
se,webnode,outrageous-injustice)/muslims 20180329175656 http://outrageous-injustice.webnode.se:80/muslims text/html 301 3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ 334
se,webnode,outrageous-injustice)/muslims 20180429215030 http://outrageous-injustice.webnode.se:80/muslims text/html 301 3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ 336
se,webnode,outrageous-injustice)/muslims 20180601151728 http://outrageous-injustice.webnode.se:80/muslims text/html 301 3I42H3S6NNFQ2MSVX7XZKYAYSCX5QBYJ 333
se,webnode,outrageous-injustice)/muslims 20180702132839 